# 📊 Data Processing, EDA & Benchmark Extraction — Help Desk AI Agent

Notebook này xử lý tập dữ liệu **1.000.000 tickets** từ Kaggle (`data/helpdesk_tickets.csv`) nhằm:
1. **Exploratory Data Analysis (EDA)**: Phân tích phân bố Category, Priority, Resolution Time & Escalation.
2. **Data Cleaning & Normalization**: Chuẩn hóa Category/Priority về định dạng của dự án Help Desk AI Agent.
3. **Stratified Benchmark Extraction**: Trích xuất 500 mẫu đại diện cân bằng phục vụ đánh giá Accuracy & F1-Score.
4. **Historical Memory Extraction**: Trích xuất ticket đã giải quyết để nạp vào RAG Vector DB (ChromaDB).
5. **Duplicate Detection Test**: Thử nghiệm tìm kiếm ticket trùng lặp dựa trên Cosine Similarity.

---
## 1. Import Thư viện & Load Dữ liệu

In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path

CSV_PATH = Path("../data/helpdesk_tickets.csv") if Path("../data/helpdesk_tickets.csv").exists() else Path("data/helpdesk_tickets.csv")

print(f"📁 Đang đọc file dataset: {CSV_PATH.resolve()}")
df = pd.read_csv(CSV_PATH, nrows=50000)
print(f"✅ Kích thước mẫu nạp: {df.shape[0]:,} dòng × {df.shape[1]} cột")
df.head()

📁 Đang đọc file dataset: C:\Users\Admin\Python Advanced\VinAI Lab\P-236\data\helpdesk_tickets.csv
✅ Kích thước mẫu nạp: 50,000 dòng × 11 cột


,Ticket_ID,Date_Created,Date_Resolved,Category,Subcategory,Priority,Status,Assigned_Team,Resolution_Time_Hrs,Description,Escalated
0,1,2023-11-16,2023-11-23,Access,Access denied by auditor policy,Low,Open,Desktop Support,3.56,User reported Access denied by auditor policy....,True
1,2,2023-11-29,2023-12-02,Security,Suspicious DLL injection,Critical,Open,Security Team,4.14,User reported Suspicious DLL injection. Furthe...,False
2,3,2023-11-29,2023-12-06,Security,Suspicious SOC alert,Low,Pending,Security Team,2.98,User reported Suspicious SOC alert. Further in...,True
3,4,2023-11-04,2023-11-09,Software,App freezing,Low,On Hold,Application Support,4.45,User reported App freezing. Further investigat...,False
4,5,2023-11-03,2023-11-10,Access,Access logging failure,Critical,In Progress,Desktop Support,3.49,User reported Access logging failure. Further ...,True


---
## 2. Thống kê & Phân tích Dữ liệu (EDA)

In [2]:
print("=== PHÂN BỐ CATEGORY ===")
print(df['Category'].value_counts())

print("\n=== PHÂN BỐ PRIORITY ===")
print(df['Priority'].value_counts())

print("\n=== PHÂN BỐ TRẠNG THÁI (STATUS) ===")
print(df['Status'].value_counts())

print("\n=== THỜI GIAN XỬ LÝ TRUNG BÌNH (GIỜ) THEO PRIORITY ===")
print(df.groupby('Priority')['Resolution_Time_Hrs'].mean().round(2))

print("\n=== TỶ LỆ LEO THANG (ESCALATED) ===")
print(df['Escalated'].value_counts(normalize=True).map(lambda x: f"{x:.1%}"))

=== PHÂN BỐ CATEGORY ===
Category
Access      10124
Security    10027
Hardware    10014
Network     10009
Software     9826
Name: count, dtype: int64

=== PHÂN BỐ PRIORITY ===
Priority
Medium      12675
High        12546
Critical    12490
Low         12289
Name: count, dtype: int64

=== PHÂN BỐ TRẠNG THÁI (STATUS) ===
Status
In Progress    10138
On Hold        10027
Resolved        9960
Pending         9941
Open            9934
Name: count, dtype: int64

=== THỜI GIAN XỬ LÝ TRUNG BÌNH (GIỜ) THEO PRIORITY ===
Priority
Critical    5.24
High        5.29
Low         5.23
Medium      5.29
Name: Resolution_Time_Hrs, dtype: float64

=== TỶ LỆ LEO THANG (ESCALATED) ===
Escalated
True     50.1%
False    49.9%
Name: proportion, dtype: str


---
## 3. Chuẩn hóa & Ánh xạ Dữ liệu (Data Mapping & Cleaning)

In [3]:
CATEGORY_MAP = {
    'Access': 'access_permission',
    'Security': 'security',
    'Software': 'software',
    'Hardware': 'hardware',
    'Network': 'network'
}

PRIORITY_MAP = {
    'Low': 'low',
    'Medium': 'medium',
    'High': 'high',
    'Critical': 'critical'
}

clean_df = df.copy()
clean_df['sys_category'] = clean_df['Category'].map(CATEGORY_MAP).fillna('other')
clean_df['sys_priority'] = clean_df['Priority'].map(PRIORITY_MAP).fillna('medium')
clean_df['title'] = clean_df.apply(lambda row: f"[{row['Category']}] {row['Subcategory']}" if pd.notna(row['Subcategory']) else f"Ticket #{row['Ticket_ID']}", axis=1)

print("✅ Chuẩn hóa dữ liệu hoàn tất.")
clean_df[['Ticket_ID', 'title', 'sys_category', 'sys_priority', 'Description']].head()

✅ Chuẩn hóa dữ liệu hoàn tất.


,Ticket_ID,title,sys_category,sys_priority,Description
0,1,[Access] Access denied by auditor policy,access_permission,low,User reported Access denied by auditor policy....
1,2,[Security] Suspicious DLL injection,security,critical,User reported Suspicious DLL injection. Furthe...
2,3,[Security] Suspicious SOC alert,security,low,User reported Suspicious SOC alert. Further in...
3,4,[Software] App freezing,software,low,User reported App freezing. Further investigat...
4,5,[Access] Access logging failure,access_permission,critical,User reported Access logging failure. Further ...


---
## 4. Trích xuất Tập Benchmark Đánh giá (500 Mẫu Cân Bằng)

In [4]:
benchmark_samples = []
for cat in clean_df['sys_category'].unique():
    cat_df = clean_df[clean_df['sys_category'] == cat]
    sample_size = min(100, len(cat_df))
    benchmark_samples.append(cat_df.sample(n=sample_size, random_state=42))

benchmark_df = pd.concat(benchmark_samples).reset_index(drop=True)
print(f"✅ Trích xuất tập Benchmark hoàn tất: {len(benchmark_df)} mẫu")
print(benchmark_df['sys_category'].value_counts())

benchmark_records = []
for _, r in benchmark_df.iterrows():
    benchmark_records.append({
        "ticket_id": int(r["Ticket_ID"]),
        "title": r["title"],
        "description": r["Description"],
        "true_category": r["sys_category"],
        "true_priority": r["sys_priority"],
        "assigned_team": r["Assigned_Team"],
        "is_escalated": bool(r["Escalated"])
    })

output_json_path = Path("../data/benchmark_kaggle_500.json") if Path("../data").exists() else Path("data/benchmark_kaggle_500.json")
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(benchmark_records, f, ensure_ascii=False, indent=2)

print(f"📄 Đã lưu file Benchmark tại: {output_json_path.resolve()}")

✅ Trích xuất tập Benchmark hoàn tất: 500 mẫu
sys_category
access_permission    100
security             100
software             100
hardware             100
network              100
Name: count, dtype: int64
📄 Đã lưu file Benchmark tại: C:\Users\Admin\Python Advanced\VinAI Lab\P-236\data\benchmark_kaggle_500.json


---
## 5. Trích xuất Dữ liệu Lịch sử (Historical Ticket Memory) nạp vào RAG

In [5]:
resolved_df = clean_df[clean_df['Status'] == 'Resolved'].head(200)

memory_entries = []
for _, r in resolved_df.iterrows():
    memory_entries.append({
        "doc_id": f"mem-kgl-{r['Ticket_ID']}",
        "title": r['title'],
        "content": f"Sự cố: {r['Description']}. Phân loại: {r['sys_category']}. Nhóm xử lý: {r['Assigned_Team']}. Thời gian giải quyết: {r['Resolution_Time_Hrs']} giờ.",
        "category": r['sys_category'],
        "solution": f"Đã được giải quyết bởi nhóm {r['Assigned_Team']} trong {r['Resolution_Time_Hrs']} giờ.",
    })

memory_path = Path("../data/historical_ticket_memory.json") if Path("../data").exists() else Path("data/historical_ticket_memory.json")
with open(memory_path, "w", encoding="utf-8") as f:
    json.dump(memory_entries, f, ensure_ascii=False, indent=2)

print(f"🧠 Đã trích xuất {len(memory_entries)} bài học lịch sử tại: {memory_path.resolve()}")

🧠 Đã trích xuất 200 bài học lịch sử tại: C:\Users\Admin\Python Advanced\VinAI Lab\P-236\data\historical_ticket_memory.json


---
## 6. Thử nghiệm Phát hiện Ticket Trùng lặp (Duplicate Detection Test)

In [6]:
try:
    from sentence_transformers import SentenceTransformer, util
    print("Loading SentenceTransformer model...")
    model = SentenceTransformer('all-MiniLM-L6-v2')

    query_text = "User reported Access denied by auditor policy. Need help urgent."
    corpus_texts = benchmark_df['Description'].tolist()

    query_emb = model.encode(query_text, convert_to_tensor=True)
    corpus_embs = model.encode(corpus_texts, convert_to_tensor=True)

    cosine_scores = util.cos_sim(query_emb, corpus_embs)[0]
    top_results = np.argsort(-cosine_scores.cpu().numpy())[:3]

    print(f"\n🔍 Query: \"{query_text}\"")
    print("\n🎯 Top 3 Ticket tương tự nhất trong CSDL:")
    for idx in top_results:
        score = cosine_scores[idx].item()
        matched_ticket = benchmark_df.iloc[idx]
        print(f"  • [Score: {score:.1%}] Ticket #{matched_ticket['Ticket_ID']} ({matched_ticket['sys_category']}): \"{matched_ticket['Description'][:80]}...\"")
except Exception as e:
    print(f"⚠️ Cần cài đặt sentence-transformers để chạy test similarity: {e}")

c:\Users\Admin\Python Advanced\VinAI Lab\P-236\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading SentenceTransformer model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14711.84it/s]



🔍 Query: "User reported Access denied by auditor policy. Need help urgent."

🎯 Top 3 Ticket tương tự nhất trong CSDL:
  • [Score: 93.2%] Ticket #33595 (access_permission): "User reported Access denied by auditor policy. Further investigation needed...."
  • [Score: 93.2%] Ticket #34187 (access_permission): "User reported Access denied by auditor policy. Further investigation needed...."
  • [Score: 75.3%] Ticket #15010 (access_permission): "User reported Access audit failure. Further investigation needed...."
